In [ ]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
WORK_DIR = "/workspace/18-4-2026"

# Clone or pull latest
if not os.path.exists(os.path.join(WORK_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", WORK_DIR, "lfs", "pull"], check=False)

# Install dependencies
subprocess.run(["pip", "install", "-q", "-r", os.path.join(WORK_DIR, "requirements.txt")], check=True)

# Set working directory to notebooks/ so Path.cwd().parent resolves to repo root
os.chdir(os.path.join(WORK_DIR, "notebooks"))

# NB4: CKA Representational Geometry (Experiment A2)

**CPU notebook** (~30 min). Compares representational geometry between legible and
illegible CoTs using Centered Kernel Alignment (CKA).

At each layer, compute linear CKA between legible and illegible activation trajectories
(resampled to fixed relative positions). Aggregate with bootstrap CIs.
Split by foreignness to check distributional-shift confound.

**Requires:** NB1 outputs (`activations/{G1,G3}_full_seq/`)

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_activations, load_foreignness_scores, linear_cka,
    resample_to_relative_positions, bootstrap_ci_metric,
    ACTIVATIONS_DIR, PHASE2_RESULTS_DIR,
)

PHASE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Checkpoint: check if final output already exists
_output_path = PHASE2_RESULTS_DIR / 'cka_results.json'
if _output_path.exists():
    import json as _json
    with open(_output_path) as _f:
        _saved = _json.load(_f)
    print(f"CACHED: {_output_path} exists.")
    for model_name in _saved:
        n_layers = len(_saved[model_name])
        print(f"  {model_name}: {n_layers} layers computed")
    print("Delete this file and re-run to recompute.")

In [ ]:
# Load full-sequence activations
g3_full = load_activations(ACTIVATIONS_DIR / "G3_full_seq")
g1_full = load_activations(ACTIVATIONS_DIR / "G1_full_seq")

# Load metadata
with open(ACTIVATIONS_DIR / "G3_full_seq" / "metadata.json") as f:
    g3_meta = json.load(f)
with open(ACTIVATIONS_DIR / "G1_full_seq" / "metadata.json") as f:
    g1_meta = json.load(f)

g3_labels = [s[3] for s in g3_meta['sample_ids']]
g1_labels = [s[3] for s in g1_meta['sample_ids']]

print(f"G3 full-seq: {len(g3_labels)} samples, layers={sorted(g3_full.keys())}")
print(f"G1 full-seq: {len(g1_labels)} samples, layers={sorted(g1_full.keys())}")
print(f"G3 labels: {dict(Counter(g3_labels))}")
print(f"G1 labels: {dict(Counter(g1_labels))}")

In [ ]:
# For each layer, resample sequences to fixed positions and compute CKA
# between legible vs illegible groups.
N_POSITIONS = 100  # Fixed number of relative positions

def compute_group_cka(full_acts, labels, layer_indices, n_positions=100):
    """Compute CKA between legible and illegible groups at each layer.
    
    For each layer:
    1. Resample all sequences to n_positions relative positions
    2. Flatten to (n_samples, n_positions * hidden_dim)
    3. Compute CKA between the two groups
    """
    legible_idx = [i for i, l in enumerate(labels) if l == 'REASONING_LEGIBLE']
    illegible_idx = [i for i, l in enumerate(labels) if l == 'ILLEGIBLE']
    
    print(f"Legible: {len(legible_idx)}, Illegible: {len(illegible_idx)}")
    
    if len(legible_idx) < 3 or len(illegible_idx) < 3:
        print("Insufficient samples for CKA")
        return {}
    
    # Use matched sample sizes (min of both)
    n_match = min(len(legible_idx), len(illegible_idx))
    
    results = {}
    for layer_idx in sorted(layer_indices):
        acts_list = full_acts[layer_idx]
        
        # Resample each sequence to fixed positions
        legible_resampled = []
        for i in legible_idx[:n_match]:
            resampled = resample_to_relative_positions(acts_list[i], n_positions)
            legible_resampled.append(resampled.flatten())
        
        illegible_resampled = []
        for i in illegible_idx[:n_match]:
            resampled = resample_to_relative_positions(acts_list[i], n_positions)
            illegible_resampled.append(resampled.flatten())
        
        X_leg = np.array(legible_resampled)
        X_ill = np.array(illegible_resampled)
        
        # CKA between groups
        cka_val = linear_cka(X_leg, X_ill)
        
        # Within-group CKA as baseline (self-similarity)
        # Split legible in half and compute CKA
        if len(legible_idx) >= 6:
            half = n_match // 2
            cka_self_leg = linear_cka(X_leg[:half], X_leg[half:2*half])
        else:
            cka_self_leg = np.nan
        
        # Bootstrap CI on CKA
        boot_ckas = []
        rng = np.random.RandomState(42)
        for _ in range(200):
            idx_l = rng.choice(n_match, size=n_match, replace=True)
            idx_i = rng.choice(n_match, size=n_match, replace=True)
            boot_cka = linear_cka(X_leg[idx_l], X_ill[idx_i])
            boot_ckas.append(boot_cka)
        boot_ckas = np.array(boot_ckas)
        ci_lo = float(np.percentile(boot_ckas, 2.5))
        ci_hi = float(np.percentile(boot_ckas, 97.5))
        
        results[layer_idx] = {
            'cka': float(cka_val),
            'cka_ci': (ci_lo, ci_hi),
            'cka_self_legible': float(cka_self_leg),
            'n_legible': n_match,
            'n_illegible': n_match,
        }
        print(f"  Layer {layer_idx}: CKA={cka_val:.3f} [{ci_lo:.3f}, {ci_hi:.3f}] "
              f"self_leg={cka_self_leg:.3f}")
    
    return results

print("=== G3 CKA ===")
g3_cka = compute_group_cka(g3_full, g3_labels, sorted(g3_full.keys()))

print("\n=== G1 CKA ===")
g1_cka = compute_group_cka(g1_full, g1_labels, sorted(g1_full.keys()))

In [ ]:
# Plot CKA across layers
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, name, cka_results in [(axes[0], 'G3 (QwQ-32B)', g3_cka),
                               (axes[1], 'G1 (R1-Distill)', g1_cka)]:
    if not cka_results:
        ax.set_title(f'{name}: insufficient data')
        continue
    
    layers = sorted(cka_results.keys())
    cka_vals = [cka_results[l]['cka'] for l in layers]
    ci_los = [cka_results[l]['cka_ci'][0] for l in layers]
    ci_his = [cka_results[l]['cka_ci'][1] for l in layers]
    self_vals = [cka_results[l]['cka_self_legible'] for l in layers]
    
    yerr_lo = [v - lo for v, lo in zip(cka_vals, ci_los)]
    yerr_hi = [hi - v for v, hi in zip(cka_vals, ci_his)]
    
    ax.errorbar(layers, cka_vals, yerr=[yerr_lo, yerr_hi],
                fmt='o-', capsize=4, label='Legible vs Illegible', linewidth=2)
    ax.plot(layers, self_vals, 's--', alpha=0.6, label='Legible self-CKA', linewidth=1.5)
    ax.set_xlabel('Layer')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Linear CKA')
fig.suptitle('Exp A2: Representational Geometry (CKA) -- Legible vs Illegible', fontsize=13)
fig.tight_layout()
fig.savefig(str(PHASE2_RESULTS_DIR / 'a2_cka_geometry.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Split by foreignness: high vs low foreignness subsets
foreignness = load_foreignness_scores()

# Get mean foreignness per sample for G3
g3_sample_foreignness = []
for s in g3_meta['sample_ids']:
    sid, gid, epoch, label = s
    scores = []
    for rid in ['R1', 'R2', 'R3']:
        key = (sid, gid, epoch, rid)
        if key in foreignness:
            scores.append(foreignness[key])
    g3_sample_foreignness.append(np.mean(scores) if scores else np.nan)

g3_sample_foreignness = np.array(g3_sample_foreignness)
valid = np.isfinite(g3_sample_foreignness)

if valid.sum() > 0:
    median_f = np.median(g3_sample_foreignness[valid])
    print(f"Median foreignness: {median_f:.2f}")
    
    # Report foreignness by label
    for label in ['REASONING_LEGIBLE', 'ILLEGIBLE']:
        mask = [l == label and v for l, v in zip(g3_labels, valid)]
        if any(mask):
            f_vals = g3_sample_foreignness[mask]
            print(f"  {label}: mean_foreignness={f_vals.mean():.2f} +/- {f_vals.std():.2f} (n={len(f_vals)})")
else:
    print("No foreignness data available for CKA split")

In [ ]:
# Save CKA results
output = {
    'G3': {int(k): v for k, v in g3_cka.items()},
    'G1': {int(k): v for k, v in g1_cka.items()},
}
with open(PHASE2_RESULTS_DIR / 'cka_results.json', 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"Saved to {PHASE2_RESULTS_DIR / 'cka_results.json'}")